### IMPLIED VOLATILITY USING LINEAR REGRESSION

Consider an European Call and Put options on an underlying asset with price assumed to be log-normal

Implied volatility is given by the volatility from the lognormal model that makes the Black-Scholes value of the option equal to the market value

$$
\begin{align}

C_{market} &= C_{BS}(S, K, T, \sigma, r, q) = S e^{-qT} \Phi(d_1) - K e^{-rT} \Phi(d_2) \\
P_{market} &= P_{BS}(S, K, T, \sigma, r, q) = K e^{-rT} \Phi(-d_2) - S e^{-qT} \Phi(-d_1) \\
d_1 &= \frac{log\left( \frac{S}{K} \right) + \left( r - q + \frac{\sigma^2}{2} \right)T}{\sigma \sqrt{T}} \\
d_2 &= d_1 - \sigma \sqrt{T}

\end{align}
$$

Note that: 
- continuous divided yields are generally not reported
- interest rates can be chosen from a set of different discount curves

Least Squares method and Put-Call parity can be used to overcome these issues and find the implied volatility using Newton's method


PUT-CALL PARITY

A Long position in a Call option and a Short position in a Put option with same strikes K and maturity T is equivalent to a Long position in a forward contract with delivery price K and maturity T

$$ 
\begin{align}
C - P &= S e^{-qT} - K e^{-rT} \\
y &= \beta_0 + K \beta_1

\end{align}
$$

We can use Least Squares to solve for $\beta_0, \beta_1$

In [18]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import scipy

df = pd.DataFrame({
    "Strike": [
        1175, 1200, 1225, 1250,
        1275, 1300, 1325, 1350,
        1375, 1400, 1425, 1450,
        1500, 1550, 1575, 1600
    ],
    "Call Price": [
        225.40, 205.65, 186.20, 167.50,
        149.15, 131.70, 115.25, 99.55,
        84.90, 71.10, 58.70, 47.25,
        29.25, 15.80, 11.10, 7.90
    ],
    "Put Price": [
        46.60, 51.55, 57.15, 63.30,
        70.15, 77.70, 86.20, 95.30,
        105.30, 116.55, 129.00, 143.20,
        173.95, 210.80, 230.90, 252.40
    ]
})

t_maturity = 199/252 # Days between March 9th and December 22

In [19]:
X = df['Strike']
X = sm.add_constant(X)
y = df['Call Price'] - df['Put Price']

model = sm.OLS(y, X)
results = model.fit()
results.summary()

beta_0 = results.params['const']
beta_1 = results.params['Strike']

We found that 
$$\begin{align}
\beta_0 &= S e^{-qT} = 1349.63 \\
\beta_1 &= -e^{-rT} = -0.9965
\end{align}$$

Note that we can use these two results as follows:
$$\begin{align}
C_{BS}(S, K, T, \sigma, r, q) &= S e^{-qT} \Phi(d_1) - K e^{-rT} \Phi(d_2) = \beta_0 \Phi(d_1) + \beta_1 \Phi(d_2) \\

P_{BS}(S, K, T, \sigma, r, q) &= K e^{-rT} \Phi(-d_2) - S e^{-qT} \Phi(-d_1) = -\beta_1 \Phi(-d_2) - \beta_0 \Phi(-d_1) \\
\end{align}$$

In addition: 
$$\begin{align}
log\left( \frac{S}{K}\right) + (r - q)T &= log\left( \frac{S}{K}\right) + log\left(e^{(r - q)T} \right) \\
                                        &= log\left( \frac{S e^{-qT}}{K e^{-rT}} \right) \\
                                        &= log\left( -\frac{\beta_0}{K \beta_1} \right) \\
\end{align}$$

Hence we get:
$$\begin{align}
d_1 &= \frac{log\left( \frac{S}{K} \right) + \left( r - q + \frac{\sigma^2}{2} \right)T}{\sigma \sqrt{T}} \\
    &= \frac{log\left( -\frac{\beta_0}{K \beta_1} \right) + \frac{\sigma^2 T}{2} }{\sigma \sqrt{T}} \\

d_2 &= \frac{log\left( -\frac{\beta_0}{K \beta_1} \right) - \frac{\sigma^2 T}{2} }{\sigma \sqrt{T}} \\

\end{align}$$

Using this method we have removed the need to know the parameters $r$, $q$, and $S$ as they are baked in the result from the OLS linear regression

Now we can use a root finding algorithm to extract the implied volatility using the functions:
$$\begin{align}
f_C(x) = C_{BS}(\beta_0, \beta_1, T, \sigma) - C_{market} = 0\\
f_P(x) = P_{BS}(\beta_0, \beta_1, T, \sigma) - P_{market} = 0
\end{align}$$

In [20]:
# Use root finding algorithm
def fun_call(sigma, b0, b1, K, t, C_market):
    # Note that b1 is defined as -e^(-rT)
    d1 = (np.log(-b0/K/b1) + 0.5 * t * sigma**2) / sigma / np.sqrt(t)
    d2 = (np.log(-b0/K/b1) - 0.5 * t * sigma**2) / sigma / np.sqrt(t)

    cdf_d1 = scipy.stats.norm.cdf(d1)
    cdf_d2 = scipy.stats.norm.cdf(d2)
    price = b0 * cdf_d1 + K * b1 * cdf_d2
    return price - C_market

# Bisection
implied_vol_call = np.zeros(len(df['Strike']))
for i,(K, C_market) in enumerate(zip(df['Strike'], df['Call Price'])):
    sol_bisect = scipy.optimize.root_scalar(
        fun_call,
        args=(beta_0, beta_1, K, t_maturity, C_market),
        bracket=[0.01, 1],
        method="bisect",
        xtol=1e-8
    )

    implied_vol_call[i] = round(sol_bisect['root'] * 100, 2)
df['Implied Vol Call'] = implied_vol_call

In [21]:
def fun_put(sigma, b0, b1, K, t, P_market):
    # Note that b1 is defined as -e^(-rT)
    d1 = (np.log(-b0/K/b1) + 0.5 * t * sigma**2) / sigma / np.sqrt(t)
    d2 = (np.log(-b0/K/b1) - 0.5 * t * sigma**2) / sigma / np.sqrt(t)

    cdf_d1 = scipy.stats.norm.cdf(-d1)
    cdf_d2 = scipy.stats.norm.cdf(-d2)
    price = -K * b1 * cdf_d2 - b0 * cdf_d1
    return price - P_market

# Bisection
implied_vol_put = np.zeros(len(df['Strike']))
for i,(K, P_market) in enumerate(zip(df['Strike'], df['Put Price'])):
    sol_bisect = scipy.optimize.root_scalar(
        fun_put,
        args=(beta_0, beta_1, K, t_maturity, P_market),
        bracket=[0.01, 1],
        method="bisect",
        xtol=1e-8
    )

    implied_vol_put[i] = round(sol_bisect['root'] * 100, 2)
df['Implied Vol Put'] = implied_vol_put

In [22]:
df

,Strike,Call Price,Put Price,Implied Vol Call,Implied Vol Put
0,1175,225.40,46.60,25.73,25.72
1,1200,205.65,51.55,24.99,24.92
2,1225,186.20,57.15,24.19,24.16
3,1250,167.50,63.30,23.44,23.40
4,1275,149.15,70.15,22.63,22.65
5,1300,131.70,77.70,21.86,21.91
6,1325,115.25,86.20,21.15,21.20
7,1350,99.55,95.30,20.41,20.43
8,1375,84.90,105.30,19.69,19.66
9,1400,71.10,116.55,18.94,18.94


Note that a consequence of the Put-Call parity is that the theoreical values of implied volatilities of calls and puts with the same strike $K$ are the same
$$\sigma_C = \sigma_P$$
Because the regression only approximates put-call parity, small differences may remain, indeed we notice that while being very close to each other they are not identical

### Using Vega
Since the derivative of the price of calls and puts under the Black-Scholes model are known, we can use the Vega and Newton's method to find the implied volatilities